In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import max, sum
data_path = "sales2.txt"

def date_parser(date: str):
    return int(date.split("/")[0])

session = SparkSession.builder.getOrCreate()
data = session.read.load(data_path, format="csv", header = False, inferSchema = True)

session.udf.register("date_parser", date_parser)
data = data.selectExpr("_c0", "date_parser(_c1) as year", "_c2 as purchases")
data.show(10)
data = data.filter("year = 2020 or year = 2021")

newData = data.select("*").groupby("_c0","year").agg(sum("purchases").alias("sumPurchase"))
maximum = newData.groupBy("year").agg(max("sumPurchase").alias("maxPurchase"))

joinedDF = newData.join(maximum, maximum.year == newData.year)
joinedDF.createOrReplaceTempView("sales")
joinedDF2 = session.sql("select _c0 from sales where sumPurchase = maxPurchase")
# joinedDF2.write.csv("outputSQL1", header=False)
newData.show(10)
maximum.show()

joinedDF.show()
joinedDF2.show()

25/12/16 18:49:45 WARN SimpleFunctionRegistry: The function date_parser replaced a previously registered function.


+----+----+---------+
| _c0|year|purchases|
+----+----+---------+
|pid1|2021|       12|
|pid1|2021|        1|
|pid1|2021|       20|
|pid1|2021|       12|
|pid2|2021|        1|
|pid2|2021|        1|
|pid2|2021|       20|
|pid2|2021|        1|
|pid3|2021|        1|
|pid3|2021|        1|
+----+----+---------+
only showing top 10 rows


25/12/16 18:49:46 WARN Column: Constructing trivially true equals predicate, 'year == year'. Perhaps you need to use aliases.


+----+----+-----------+
| _c0|year|sumPurchase|
+----+----+-----------+
|pid1|2020|         45|
|pid3|2020|         35|
|pid2|2020|         23|
|pid3|2021|         35|
|pid2|2021|         23|
|pid1|2021|         45|
+----+----+-----------+

+----+-----------+
|year|maxPurchase|
+----+-----------+
|2020|         45|
|2021|         45|
+----+-----------+

+----+----+-----------+----+-----------+
| _c0|year|sumPurchase|year|maxPurchase|
+----+----+-----------+----+-----------+
|pid2|2020|         23|2020|         45|
|pid3|2020|         35|2020|         45|
|pid1|2020|         45|2020|         45|
|pid1|2021|         45|2021|         45|
|pid2|2021|         23|2021|         45|
|pid3|2021|         35|2021|         45|
+----+----+-----------+----+-----------+

+----+
| _c0|
+----+
|pid1|
|pid1|
+----+

